# 02 - Stage 1 Synthetic Data Registry

This notebook registers the already-generated DCGAN synthetic images as the official output of Stage 1: Synthetic Lung Radiography Image Generation.

It does not train a GAN. It verifies the synthetic image folder, creates a manifest, writes metadata, and saves a sample grid for later experiment tracking.

## 1. Mount Google Drive

Mount Drive so the notebook can read your generated DCGAN images and optionally keep outputs persistent.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Or Pull The Repository

If the repository already exists in the runtime, pull the latest changes. Otherwise, clone it.

In [ ]:
from pathlib import Path
import os

REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')

if REPO_ROOT.exists():
    print(f'Repository exists at {REPO_ROOT}. Pulling latest changes...')
    %cd {REPO_ROOT}
    !git pull
else:
    print(f'Cloning repository to {REPO_ROOT}...')
    %cd /content
    !git clone {REPO_URL}
    %cd {REPO_ROOT}

print('Current working directory:', Path.cwd())

## 3. Install Lightweight Dependencies

The registry uses standard Python plus `pandas`, `PyYAML`, `Pillow`, and `matplotlib`. These are lightweight compared with training dependencies.

In [ ]:
!pip install -q pandas PyYAML Pillow matplotlib

## 4. Editable Paths

Set `SYNTHETIC_DIR` to the DCGAN synthetic image folder in Google Drive. This folder is expected to contain the class folders `COVID`, `Lung_Opacity`, `Viral_Pneumonia`, and `Normal`.

The generated registry artifacts are written under the repository paths required by the project:

- `data/manifests/synthetic_dcgan.csv`
- `results/stage1_synthesis/dcgan_metadata.yaml`
- `results/stage1_synthesis/synthetic_summary.json`
- `results/stage1_synthesis/sample_grid.png`

In [ ]:
# Edit this path for your Google Drive layout.
SYNTHETIC_DIR = Path('/content/drive/MyDrive/path/to/DCGAN_synthetic_images')

MANIFEST_PATH = REPO_ROOT / 'data/manifests/synthetic_dcgan.csv'
STAGE1_DIR = REPO_ROOT / 'results/stage1_synthesis'
METADATA_PATH = STAGE1_DIR / 'dcgan_metadata.yaml'
SUMMARY_PATH = STAGE1_DIR / 'synthetic_summary.json'
SAMPLE_GRID_PATH = STAGE1_DIR / 'sample_grid.png'

CLASS_TO_LABEL = {
    'COVID': 0,
    'Lung_Opacity': 1,
    'Viral_Pneumonia': 2,
    'Normal': 3,
}

print('REPO_ROOT:', REPO_ROOT)
print('SYNTHETIC_DIR:', SYNTHETIC_DIR)
print('MANIFEST_PATH:', MANIFEST_PATH)
print('STAGE1_DIR:', STAGE1_DIR)

## 5. Verify Synthetic Folder

This check confirms that the DCGAN folder exists, class folder names match exactly, and each class contains images.

In [ ]:
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def is_image(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS

def class_image_dir(root: Path, class_name: str) -> Path:
    class_root = root / class_name
    image_subdir = class_root / 'images'
    return image_subdir if image_subdir.exists() else class_root

def find_images(root: Path):
    if not root.exists():
        return []
    return sorted(path for path in root.rglob('*') if is_image(path))

if not SYNTHETIC_DIR.exists():
    print(f'CRITICAL: SYNTHETIC_DIR does not exist: {SYNTHETIC_DIR}')
else:
    present = sorted(path.name for path in SYNTHETIC_DIR.iterdir() if path.is_dir())
    expected = list(CLASS_TO_LABEL)
    missing = [name for name in expected if name not in present]
    unexpected = [name for name in present if name not in expected]

    print('Present folders:', present)
    print('Missing folders:', missing)
    print('Unexpected folders:', unexpected)

    class_counts = {}
    for class_name in expected:
        class_counts[class_name] = len(find_images(class_image_dir(SYNTHETIC_DIR, class_name)))

    print('Class counts:')
    for class_name, count in class_counts.items():
        print(f'  {class_name}: {count}')
    print('TOTAL:', sum(class_counts.values()))

## 6. Display Synthetic Samples

Preview synthetic images from each class before registering the manifest.

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt

def display_images(paths, title, max_images=4):
    paths = [Path(p) for p in paths if Path(p).exists()]
    if not paths:
        print(f'WARNING: No images available for {title}')
        return
    sample = paths[:max_images]
    fig, axes = plt.subplots(1, len(sample), figsize=(4 * len(sample), 4))
    if len(sample) == 1:
        axes = [axes]
    for ax, path in zip(axes, sample):
        image = Image.open(path).convert('RGB')
        ax.imshow(image, cmap='gray')
        ax.set_title(path.name[:32])
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

random.seed(42)
for class_name in CLASS_TO_LABEL:
    images = find_images(class_image_dir(SYNTHETIC_DIR, class_name))
    random.shuffle(images)
    display_images(images, f'DCGAN synthetic: {class_name}', max_images=4)

## 7. Register Stage 1 Output

Run the registry script. It writes the manifest, metadata, summary JSON, and sample grid. Later `COVID-QU-Syn` experiments should consume `data/manifests/synthetic_dcgan.csv`.

In [ ]:
%cd {REPO_ROOT}

!python scripts/register_synthetic_dataset.py \
  --synthetic-dir "{SYNTHETIC_DIR}" \
  --manifest-path "{MANIFEST_PATH}" \
  --metadata-path "{METADATA_PATH}" \
  --summary-path "{SUMMARY_PATH}" \
  --sample-grid-path "{SAMPLE_GRID_PATH}" \
  --generator DCGAN \
  --source stage1_synthesis

## 8. Inspect Registry Artifacts

Load the generated files and verify that the manifest is ready for downstream experiments.

In [ ]:
import json
import pandas as pd
import yaml

manifest = pd.read_csv(MANIFEST_PATH)
summary = json.loads(SUMMARY_PATH.read_text())
metadata = yaml.safe_load(METADATA_PATH.read_text())

print('Manifest rows:', len(manifest))
display(manifest.head())

print('Summary:')
print(json.dumps(summary, indent=2))

print('Metadata keys:', list(metadata.keys()))

## 9. Display Saved Sample Grid

This image can be used as the Stage 1 visual reference for registered DCGAN output.

In [ ]:
if SAMPLE_GRID_PATH.exists():
    image = Image.open(SAMPLE_GRID_PATH).convert('RGB')
    plt.figure(figsize=(12, 12))
    plt.imshow(image)
    plt.axis('off')
    plt.show()
else:
    print('WARNING: sample grid was not created:', SAMPLE_GRID_PATH)

## 10. Final Status

This confirms whether Stage 1 DCGAN output is registered for the later 12-experiment pipeline.

In [ ]:
required_outputs = [MANIFEST_PATH, METADATA_PATH, SUMMARY_PATH, SAMPLE_GRID_PATH]
for path in required_outputs:
    print(f'{path}:', 'OK' if path.exists() else 'MISSING')

if all(path.exists() for path in required_outputs):
    print('\nOK: Stage 1 DCGAN synthetic output is registered.')
    print('Use this manifest for COVID-QU-Syn and ImageNet -> COVID-QU-Syn experiments:')
    print(MANIFEST_PATH)
else:
    print('\nWARNING: Some Stage 1 registry artifacts are missing.')